# Python reference 1: values, numbers and conversions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-01-values-and-numbers.ipynb)

Identity vs equality, `type` and `isinstance`, integer and float arithmetic, rounding, `Decimal` and `Fraction`, the `math` module, truthiness and `None`, and type conversions. Practise it in [Python practice 1: values, numbers and conversions](../practice/python-practice-01-values-and-numbers.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [Names, objects and identity](#names-objects-and-identity) | [Assignment, identity and equality](#assignment-identity-and-equality), [`type()` and `isinstance()`](#type-and-isinstance) |
| [Numbers](#numbers) | [`int`](#int), [`float` and the floating-point trap](#float-and-the-floating-point-trap), [`round()`](#round), [`decimal.Decimal`](#decimaldecimal), [`fractions.Fraction`](#fractionsfraction), [The `math` module](#the-math-module) |
| [Booleans and None](#booleans-and-none) | [Truthiness](#truthiness), [`None`](#none) |
| [Conversions](#conversions) | [`int()`, `float()`, `str()`, `bool()`](#int-float-str-bool), [`repr()` vs `str()`](#repr-vs-str) |

## Names, objects and identity

### Assignment, identity and equality

Assignment binds a **name** to an **object**; it never copies. Two names can refer to the same object.

| expression | compares | use for |
| --- | --- | --- |
| `a == b` | values (calls `__eq__`) | almost everything |
| `a is b` | identity: the same object in memory | `x is None`, sentinels, and checking for aliasing |
| `id(a)` | the object's identity number | debugging aliasing |

**Notes**
- CPython caches small integers (-5 to 256) and some strings, so `is` can appear to work on values; never rely on it.
- Mutating an object through one name is visible through every other name bound to it (see
  [aliasing](python-reference-03-lists-dicts-sets.ipynb#copying-aliasing-and-mutability)).

In [ ]:
a = [1, 2]
b = a              # same object
c = [1, 2]         # equal value, different object
print(a == c, a is c, a is b)
b.append(3)
print("a sees the change:", a)
x, y = 1000, int("1000")
print("equal:", x == y, "| same object:", x is y)

### `type()` and `isinstance()`

`type(x)` returns the exact class; `isinstance(x, cls_or_tuple)` also accepts subclasses and a tuple of types. Prefer
`isinstance` for checks.

**Notes:** `bool` is a subclass of `int`, so `isinstance(True, int)` is `True` and `True + True == 2`. Check for `bool`
first when both are possible.

In [ ]:
for value in [1, 1.5, True, "1", None, [1]]:
    print(f"{value!r:6} type={type(value).__name__:8} int-like={isinstance(value, int)} number={isinstance(value, (int, float)) and not isinstance(value, bool)}")

## Numbers

### `int`

Integers have **arbitrary precision**: no overflow, only memory limits.

| operation | result | note |
| --- | --- | --- |
| `7 / 2` | `3.5` | true division always returns `float` |
| `7 // 2`, `-7 // 2` | `3`, `-4` | floor division rounds toward **minus infinity** |
| `7 % 3`, `-7 % 3` | `1`, `2` | the remainder has the sign of the divisor |
| `divmod(-7, 3)` | `(-3, 2)` | `(a // b, a % b)` in one call |
| `2 ** 100` | a 31-digit int | no overflow |
| `int("ff", 16)`, `int("0b101", 0)` | `255`, `5` | parse in any base; base 0 reads the prefix |
| `1_000_000` | `1000000` | underscores for readability |
| `x.bit_length()`, `&`, `\|`, `^`, `<<`, `>>`, `~` | bit operations | flags and masks |

**Notes:** to truncate toward zero (like C or SQL), use `int(a / b)` for small values or `-(-a // b)` patterns
carefully; `math.trunc` truncates a float.

In [ ]:
print(7 // 2, -7 // 2, 7 % 3, -7 % 3, divmod(-7, 3))
print(2 ** 100)
print(int("ff", 16), int("0b101", 0), hex(255), bin(5), (255).bit_length())
print("truncate toward zero:", int(-7 / 2), "| floor:", -7 // 2)

### `float` and the floating-point trap

`float` is an IEEE 754 double: about 15–17 significant decimal digits, stored in binary. Most decimal fractions
(0.1, 0.2) cannot be represented exactly, so arithmetic accumulates tiny errors.

| tool | use |
| --- | --- |
| `math.isclose(a, b, rel_tol=1e-9, abs_tol=0.0)` | compare floats; set `abs_tol` when values can be near zero |
| `math.fsum(values)` | an accurately rounded sum (since Python 3.12 the built-in `sum()` also compensates for float error; a `+=` loop does not) |
| `float("inf")`, `float("nan")`, `math.isinf`, `math.isnan` | special values |
| `sys.float_info` | limits and epsilon |

**Notes:** `nan != nan`, so `x == x` is `False` for NaN and `nan in [nan]` relies on identity. Never compare floats
with `==` unless they came from the same exact computation; never use them for money.

In [ ]:
import math
import sys

print(0.1 + 0.2, 0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3))
running = 0.0
for _ in range(10):
    running += 0.1
print("running +=:", running, "| sum():", sum([0.1] * 10), "| fsum:", math.fsum([0.1] * 10))
nan = float("nan")
print("nan == nan:", nan == nan, "| isnan:", math.isnan(nan), "| epsilon:", sys.float_info.epsilon)
print("isclose near zero:", math.isclose(1e-12, 0.0), math.isclose(1e-12, 0.0, abs_tol=1e-9))

### `round()`

`round(number, ndigits=None)` rounds to `ndigits` decimals; with no `ndigits` it returns an `int`. It uses **round half
to even** (banker's rounding): `round(0.5) == 0`, `round(1.5) == 2`, `round(2.5) == 2`.

**Notes:** `round(2.675, 2)` gives `2.67`, not `2.68`, because 2.675 is stored as 2.67499999… Round money with
`Decimal.quantize` and an explicit rounding mode.

In [ ]:
print([round(x) for x in (0.5, 1.5, 2.5, 3.5)], round(2.675, 2), round(1234.5678, -2))

### `decimal.Decimal`

Exact decimal arithmetic with a configurable precision and rounding mode: the right type for money.

| operation | example | note |
| --- | --- | --- |
| construct | `Decimal("19.99")` | **from a string**; `Decimal(0.1)` captures the float's binary error |
| quantize | `d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)` | round to a fixed exponent with an explicit mode |
| rounding modes | `ROUND_HALF_UP`, `ROUND_HALF_EVEN`, `ROUND_DOWN`, `ROUND_CEILING`, … | finance rules usually name one |
| context | `decimal.getcontext().prec = 28` | significant digits for arithmetic |
| mix with int | `Decimal("1.5") * 2` | allowed |
| mix with float | `Decimal("1.5") * 2.0` | **`TypeError`** |

**Notes:** `Decimal("1.10") == Decimal("1.1")` is `True` but they print differently; normalize with `quantize` before
display or comparison of strings.

In [ ]:
from decimal import ROUND_HALF_EVEN, ROUND_HALF_UP, Decimal

print(Decimal(0.1), "|", Decimal("0.1"))
prices = [Decimal("19.99"), Decimal("5.01"), Decimal("0.10")]
total = sum(prices, Decimal("0"))
print("total:", total, "| with tax:", (total * Decimal("1.2")).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))
print("2.665 half up:", Decimal("2.665").quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),
      "| half even:", Decimal("2.665").quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN))
try:
    Decimal("1.5") * 2.0
except TypeError as error:
    print("TypeError:", error)

### `fractions.Fraction`

Exact rational numbers: `Fraction(1, 3) + Fraction(1, 6) == Fraction(1, 2)`. Use for exact ratios (shares, unit
conversions) where neither float error nor fixed decimal places are acceptable. `Fraction("0.1")` parses text exactly;
`.limit_denominator(n)` approximates.

In [ ]:
from fractions import Fraction

print(Fraction(1, 3) + Fraction(1, 6), Fraction("0.1") * 3, Fraction(0.1).limit_denominator(100))

### The `math` module

| function | returns |
| --- | --- |
| `math.floor(x)`, `math.ceil(x)`, `math.trunc(x)` | int toward −∞, +∞, 0 |
| `math.fsum(it)`, `math.prod(it)` | accurate float sum; product |
| `math.isqrt(n)`, `math.sqrt(x)`, `math.pow(x, y)` | integer square root; float root and power |
| `math.gcd(*ints)`, `math.lcm(*ints)` | greatest common divisor, least common multiple |
| `math.log(x, base)`, `math.log10`, `math.log2`, `math.exp` | logarithms and exponent |
| `math.isclose`, `math.isfinite`, `math.isnan`, `math.isinf` | float checks |
| `math.inf`, `math.nan`, `math.pi`, `math.e` | constants |

In [ ]:
print(math.floor(-2.5), math.ceil(-2.5), math.trunc(-2.5), math.isqrt(99), math.gcd(12, 18), math.lcm(4, 6), math.prod([2, 3, 4]))

## Booleans and None

### Truthiness

Every object is truthy or falsy in `if`, `while`, `and`, `or`, `not` and `bool()`.

| falsy | truthy |
| --- | --- |
| `False`, `None` | `True` |
| `0`, `0.0`, `Decimal(0)`, `0j` | any non-zero number |
| `""`, `b""` | any non-empty string (including `"False"`, `"0"`, `" "`) |
| `[]`, `()`, `{}`, `set()`, `range(0)` | any non-empty container |

`a and b` returns `a` if `a` is falsy, else `b`; `a or b` returns `a` if `a` is truthy, else `b`: they return
**operands**, not booleans.

**Notes:** `value or default` replaces **every** falsy value, so `0` and `""` become the default. Use
`default if value is None else value` when only `None` means missing.

In [ ]:
for value in [0, "", "0", "False", [], [0], None]:
    print(f"{value!r:8} -> {bool(value)}")
qty = 0
print("qty or 1:", qty or 1, "| None check:", 1 if qty is None else qty)
print("and/or return operands:", [] or "fallback", "x" and "y")

### `None`

`None` is the single instance of `NoneType`: "no value". Test it with `is None` / `is not None` (not `== None`, which a
class can override). Functions without a `return` return `None`.

**Use it for:** missing values, optional arguments (`def f(limit=None)`), and "not found" results. When `None` is
itself a valid value, use a private sentinel (`_MISSING = object()`).

In [ ]:
_MISSING = object()

def lookup(mapping, key, default=_MISSING):
    if key in mapping:
        return mapping[key]
    if default is _MISSING:
        raise KeyError(key)
    return default

config = {"timeout": None}
print(lookup(config, "timeout"), lookup(config, "retries", 3))

## Conversions

### `int()`, `float()`, `str()`, `bool()`

| call | result | note |
| --- | --- | --- |
| `int("42")`, `int(" 42\n")` | `42` | surrounding whitespace is allowed |
| `int("4.2")`, `int("")`, `int("1,000")` | `ValueError` | parse to `float` or `Decimal` first, strip separators |
| `int(4.9)`, `int(-4.9)` | `4`, `-4` | truncates toward zero |
| `float("1e3")`, `float("nan")`, `float(" 2.5 ")` | `1000.0`, `nan`, `2.5` | |
| `str(1.0)`, `str(Decimal("1.10"))` | `"1.0"`, `"1.10"` | |
| `bool("False")`, `bool("0")` | `True` | any non-empty string is truthy; parse explicitly |

**Notes:** `str.isdigit()` is not a safe "is it an int" test: it rejects `"-5"` and `" 5"` but accepts superscripts like
`"²"`. Use `try: int(s) except ValueError`.

In [ ]:
def try_int(text):
    try:
        return int(text)
    except (TypeError, ValueError):
        return None

for text in ["42", " 42\n", "-5", "4.2", "", "1,000", "²", None]:
    print(f"{text!r:8} isdigit={str(str(text).isdigit()):5} int={try_int(text)}")

### `repr()` vs `str()`

`str(x)` is for people; `repr(x)` is unambiguous, ideally valid Python that recreates the value. Log and debug with
`repr` (or the `!r` f-string conversion) so `"1"` and `1`, or trailing spaces, are visible.

In [ ]:
for value in ["1", 1, "a ", Decimal("1.10")]:
    print(f"str={value!s:6} repr={value!r}")

**Next:** practise these entries in [Python practice 1: values, numbers and conversions](../practice/python-practice-01-values-and-numbers.ipynb).